In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "artifact"
ARTIFACT_DIR.mkdir(exist_ok=True)
path = ARTIFACT_DIR

file_path = ARTIFACT_DIR / "ml_df.parquet"
ml_df = pd.read_parquet(file_path)

ml_df['actual_carrier_delay_days'] = ml_df['order_delivered_customer_date'] - ml_df['order_purchase_timestamp']
print(ml_df.shape)
ml_df.columns


In [ ]:
# BASIC CLEANING 

# remove duplicates if any
ml_df = ml_df.drop_duplicates()

# convert datetime columns
datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date"
]

for col in datetime_cols:
    if col in ml_df.columns:
        ml_df[col] = pd.to_datetime(ml_df[col], errors="coerce")

In [ ]:
# Target Separation  


timedelta_cols = ["order_delivery_delay", "actual_carrier_delay_days"]
for col in timedelta_cols:
    if col in ml_df.columns and pd.api.types.is_timedelta64_dtype(ml_df[col]):
        ml_df[col] = ml_df[col].dt.total_seconds() / 86400  # → float days
y_reg = ml_df["order_delivery_delay"]

# Example:
# 1 = delayed
# 0 = on-time

ml_df["is_delayed"] = (ml_df["order_delivery_delay"] > 0).astype(int)

y_clf = ml_df["is_delayed"]



In [ ]:
ml_df.columns

**Problem Faced in early feature engineering pipeline ?**


Problem Description:------------------------



```python 
RAW TABLES
    ↓

CLEAN TABLES
    ↓

BASE TABLE -order_item level granularity DATASET (ml_df)
    ↓
ENTITY FEATURE TABLES
(customer/seller/product/order/logistics/category)
    ↓

FEATURE ASSEMBLY
    ↓

FINAL TRAINING DATASET
```

In [ ]:

def rolling_mean(df, group_col, value_col, window):
    return (
        df.groupby(group_col)[value_col]
        .transform(
            lambda x: x.shift().rolling(window=window, min_periods=1).mean()
        )
    )
 
def rolling_count(df, group_col, window):
    """Counts how many non-null past rows exist within the rolling window."""
    return (
        df.groupby(group_col)[group_col]
        .transform(
            lambda x: x.shift().rolling(window=window, min_periods=1).count()
        )
    )

In [ ]:
# Entity Based feature engineering 

## First sort merged dataset with respect order timestamp 
ml_df = ml_df.sort_values(
    by='order_purchase_timestamp'
).reset_index(drop=True)

## The Merged final table ml_df Granularity is order_item level 


## features are of different level of granularity 

ml_df['product_volume']=ml_df['product_length_cm']*ml_df['product_width_cm']*ml_df['product_height_cm']
## ORDERS ENTITY TABLE 
order_table = (
    ml_df.groupby('order_id')
    .agg(
        order_purchase_timestamp=('order_purchase_timestamp','first'),

        total_order_value=('price','sum'),

        total_freight=('freight_value','sum'),

        num_items=('product_id','count'),

        num_sellers=('seller_id','nunique'),

        num_categories=('product_category_name','nunique'),

        total_weight=('product_weight_g','sum'),
        
          total_volume=('product_volume','sum')

    )
    .reset_index()
)




In [ ]:
## TIME INTELLIGENT FEATURES ( These are order level features )

## that means for each unique order(so granularity order level wise ) they will be unique 

## Order purchase Hour
order_table['purchase_hour'] = (
    order_table['order_purchase_timestamp'].dt.hour
)

## Order Purchase Day 
order_table['purchase_day'] = (
    order_table['order_purchase_timestamp'].dt.day
)

## Order Purchase month 
order_table['purchase_month'] = (
    order_table['order_purchase_timestamp'].dt.month
)

## Order Purchase Weekday 
order_table['purchase_weekday'] = (
    order_table['order_purchase_timestamp'].dt.weekday
)
 
## Is order Purchased on weekday
order_table['is_weekend'] = (
    order_table['purchase_weekday'] >= 5
).astype(int)

# Month end flag
order_table['is_month_end'] = (
    order_table['order_purchase_timestamp']
    .dt.is_month_end
).astype(int)

# Month start flag
order_table['is_month_start'] = (
    order_table['order_purchase_timestamp']
    .dt.is_month_start
).astype(int)

## order Purchase Quarter
order_table['purchase_quarter'] = (
    order_table['order_purchase_timestamp'].dt.quarter
)

## Year end
order_table['is_year_end'] = (
    order_table['order_purchase_timestamp']
    .dt.is_year_end
).astype(int)

## historical features 

import holidays 

## Since dataset is from Brazil E commerce 
brazil_holidays=holidays.Brazil()

order_table['is_holiday'] = (
    order_table['order_purchase_timestamp']
    .dt.date
    .isin(brazil_holidays)
).astype(int)



## derived  order level features 



## Average item price in a order 
order_table['avg_item_price'] = (
    order_table['total_order_value']
    /
    order_table['num_items']
)

## freight Precentage 

order_table['freight_percentage'] = (
    order_table['total_freight']
    /
    order_table['total_order_value']
)

## Whether inside the order Product are from Multi-seller (Multi seller flag )

order_table['is_multi_seller'] = (
    order_table['num_sellers'] > 1
).astype(int)

## order Total volume 

order_table['total_volume'] = (
    ml_df.groupby('order_id')['product_volume']
    .sum()
    .values
)

## heavy Order Flag 

threshold =50

order_table['is_heavy_order'] = (
    order_table['total_weight'] > threshold
).astype(int)

In [ ]:
## Product Table 

## Granularity -order item level

product_window=30

product_table = (
    ml_df[
        [
            'product_id',
            'order_id',
            'order_purchase_timestamp',
            'price',
            'freight_value',
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm',
            'actual_carrier_delay_days'
        ]
    ]
    .sort_values(
        ['order_purchase_timestamp','order_id']
    )
    .copy()
)

In [ ]:
## product Level features 

# previous sales(entire )product wise 

product_table["product_previous_sales"] = (
    product_table.groupby("product_id").cumcount()
)

## product wise average  price - rolling average  in a window 

product_table["product_rolling_avg_price"] = rolling_mean(
    product_table, "product_id", "price", product_window
)
## product wise average freight  - rolling average  in a window 

product_table["product_rolling_avg_freight"] = rolling_mean(
    product_table, "product_id", "freight_value",product_window
)

## ## product wise average  carrier delay  - rolling average  in a window 

product_table["product_rolling_avg_carrier_delay"] = rolling_mean(
    product_table, "product_id", "actual_carrier_delay_days", product_window
)

# Cold-start flag: model knows when history is thin
product_table["product_has_history"] = (
    product_table["product_previous_sales"] > 0
).astype(int)
 
# volume 

product_table['product_volume'] = (
    product_table['product_length_cm']
    *
    product_table['product_height_cm']
    *
    product_table['product_width_cm']
)

# Density

product_table['weight_per_volume'] = (
    product_table['product_weight_g']
    /
    product_table['product_volume']
)

In [ ]:
## CATEGORY TABLE 

category_window = 50  

category_table = (
    ml_df[
        [
            'product_category_name',
            'order_id',
            'order_purchase_timestamp',
            'actual_carrier_delay_days',
            'order_delivery_delay',
            'freight_value'
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ['order_purchase_timestamp','order_id']
    )
    .copy()
)

In [ ]:
## Product Category Level Features

## Category historical average delay
category_table["category_rolling_avg_delay"] = rolling_mean(
    category_table, "product_category_name", "order_delivery_delay", category_window
)

## Category historical average carrier/delivery time
category_table["category_rolling_avg_carrier_delay"] = rolling_mean(
    category_table, "product_category_name", "actual_carrier_delay_days", category_window
)

## Category historical average freight
category_table["category_rolling_avg_freight"] = rolling_mean(
    category_table, "product_category_name", "freight_value", category_window
)


In [ ]:
# SELLER TABLE 

seller_window =20
seller_table = (
    ml_df[
        [
            'seller_id',
            'order_id',
            'order_purchase_timestamp',
            'freight_value',
            'order_delivery_delay',
            'actual_carrier_delay_days'
        ]
    ]
    .drop_duplicates(
        subset=['seller_id','order_id']
    )
    .sort_values(
        ['order_purchase_timestamp','order_id']
    )
    .copy()
)

In [ ]:
## SELLER LEVEL FEATURES 

# previous orders seller wise 

seller_table['seller_previous_orders'] = (
    seller_table
    .groupby('seller_id')
    .cumcount()
)

# historical avg delivery delay seller wise 

seller_table["seller_rolling_avg_delay"] = rolling_mean(
    seller_table, "seller_id", "order_delivery_delay", seller_window
)
# historical avg freight seller wise 

seller_table["seller_rolling_avg_freight"] = rolling_mean(
    seller_table, "seller_id", "freight_value", seller_window
)

# historical carrier delay seller wise 

seller_table["seller_rolling_avg_carrier_delay"] = rolling_mean(
    seller_table, "seller_id", "actual_carrier_delay_days", seller_window
)

seller_table["seller_has_history"] = (
    seller_table["seller_previous_orders"] > 0
).astype(int)
 

In [ ]:
## CUSTOMER TABLE 

customer_window=5

customer_table = (
    ml_df[
        [
            'customer_unique_id',
            'order_id',
            'order_purchase_timestamp',
            'price',
            'order_delivery_delay'
        ]
    ]
    .drop_duplicates(
        subset=['customer_unique_id','order_id']
    )
    .sort_values(
        ['order_purchase_timestamp','order_id']
    )
    .copy()
)

In [ ]:
## CUSTOMER LEVEL FEATURES 


# previous orders customer wise 

customer_table['customer_previous_orders'] = (
    customer_table
    .groupby('customer_unique_id')
    .cumcount()
)

# historical avg delay customer wise 

customer_table["customer_rolling_avg_delay"] = rolling_mean(
    customer_table, "customer_unique_id", "order_delivery_delay", customer_window
)

# historical avg order value customer wise 

customer_table["customer_rolling_avg_order_value"] = rolling_mean(
    customer_table, "customer_unique_id", "price", customer_window
)
# days since last order customer wise 

customer_table['customer_days_since_last_order'] = (
    customer_table
    .groupby('customer_unique_id')['order_purchase_timestamp']
    .diff()
    .dt.days
)
## flag for History available or not 
customer_table["customer_has_history"] = (
    customer_table["customer_previous_orders"] > 0
).astype(int)
 

In [ ]:
## Base Table Creation 

base_cols = [
    "order_id",
    "seller_id",
    "product_id",
    "product_category_name",
    "customer_unique_id",
    "order_purchase_timestamp",
    "price",
    "freight_value",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_volume",
    "weight_per_volume",  # already in product_table but useful raw
    "order_delivery_delay",
    "is_delayed",
    "actual_carrier_delay_days",
]

# Keep only columns that actually exist (safety guard)
base_cols = [c for c in base_cols if c in ml_df.columns]

final_ml_df = ml_df[base_cols].copy()
 

In [ ]:
## Merging Features in final table 

## merging customer level  features 
final_ml_df = final_ml_df.merge(
    customer_table[
        [
            'order_id',
            'customer_previous_orders',
            "customer_rolling_avg_delay",
        "customer_rolling_avg_order_value",
            'customer_days_since_last_order',
            'customer_has_history'
        ]
    ],
    on='order_id',
    how='left'
)

## Merging Seller Level features 

final_ml_df = final_ml_df.merge(
    seller_table[
        [
            'order_id',
            'seller_id',
            'seller_previous_orders',
            'seller_rolling_avg_delay',
        'seller_rolling_avg_freight',
        'seller_rolling_avg_carrier_delay',
        'seller_has_history',
        ]
    ],
    on=['order_id','seller_id'],
    how='left'
)
## Merging Product Level features 

final_ml_df = final_ml_df.merge(
    product_table[[
        "order_id",
        "product_id",
        "product_previous_sales",
        "product_rolling_avg_price",
        "product_rolling_avg_freight",
        "product_rolling_avg_carrier_delay",
        "product_has_history",
    ]],
    on=["order_id", "product_id"],
    how="left",
)

## Merging Category Level Features 
final_ml_df = final_ml_df.merge(
    category_table[[
        "order_id",
        "product_category_name",
        "category_rolling_avg_delay",
        "category_rolling_avg_carrier_delay",
        "category_rolling_avg_freight",
    ]],
    on=["order_id", "product_category_name"],
    how="left",
)


## Merging Order Level features 
final_ml_df = final_ml_df.merge(
    order_table.drop(columns=["order_purchase_timestamp"]),  # already in final_ml_df
    on="order_id",
    how="left",
)

In [ ]:
 
print(f"\nFinal ML DataFrame shape: {final_ml_df.shape}")
print(f"Columns: {list(final_ml_df.columns)}")

In [ ]:
## Save intermediate feature dataset

file_path = ARTIFACT_DIR / "final_ml_df.parquet"
print(file_path)
final_ml_df.to_parquet(
    file_path,
    engine="pyarrow",
    compression="snappy",
    index=False,
)
print("final_ml_df saved successfully")


In [ ]:
# ## Missing value Check 

# null_summary = final_ml_df.isnull().sum()
# null_summary = null_summary[null_summary > 0].sort_values(ascending=False)
# print(f"\nNull counts (non-zero only):\n{null_summary}")
final_ml_df.isnull().sum().sort_values(ascending=False)

## Dealing with Missing Values 


## Category 1:

# CUSTOMERS  — Rolling/historical features : 
```python
customer_historical_avg_delay
customer_historical_avg_order_value
customer_days_since_last_order
```
missing 

`171,908 ` out of `181,937`

WHY?
Because: most customers only ordered once.

so 
`shift()` make first odrer NaN, No history exists yet. This is:
```python
NORMAL
NOT bad data
```
that is why we did keep cold start customers  for first time track 
# PRODUCT -Rolling/historical features 
```python 
product_rolling_avg_carrier_delay      39716
product_rolling_avg_freight            39716
product_rolling_avg_price              39716
```
almost `39k`  This means:many products first time seen

## SELLERS -ROLLING/Historical features 
```python 
seller_rolling_avg_delay                6781
seller_rolling_avg_carrier_delay        6781
seller_rolling_avg_freight              6781

```
almost `6k` very low compared to dataset size implying most of the sellers has enough history 


## CATEGORY -ROLLING/Historical features 
```python 
category_rolling_avg_delay              2669
category_rolling_avg_freight            2669

```
almost `2.6k` very low Means categories have rich history.

## PRODUCT DIMENSIONS 

```python 
product_weight_g                          30
product_volume                            30
product_width_cm                          30
product_height_cm                         30
product_length_cm                         30
```
very tiny 


In [ ]:
## Cold-start rolling feature columns
## Missing values are filled after the temporal split using train-only medians.
cold_start_cols = [
    "customer_rolling_avg_delay",
    "customer_rolling_avg_order_value",
    "seller_rolling_avg_delay",
    "seller_rolling_avg_freight",
    "seller_rolling_avg_carrier_delay",
    "product_rolling_avg_price",
    "product_rolling_avg_freight",
    "product_rolling_avg_carrier_delay",
    "category_rolling_avg_delay",
    "category_rolling_avg_carrier_delay",
    "category_rolling_avg_freight",
]


In [ ]:
## Product metadata columns
## Missing values are filled after the temporal split using train-only medians.
dim_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_width_cm",
    "product_height_cm",
]


In [ ]:
## Product category nulls
if "product_category_name" in final_ml_df.columns:
    final_ml_df["product_category_name"] = final_ml_df["product_category_name"].fillna("unknown")


In [ ]:
# ----------  days since last order ----------
# Sentinel = 999 means "first-time customer / never ordered before"
if "customer_days_since_last_order" in final_ml_df.columns:
    final_ml_df["customer_days_since_last_order"] = (
        final_ml_df["customer_days_since_last_order"].fillna(999)
    )

In [ ]:
null_after = final_ml_df.isnull().sum()
null_after = null_after[null_after > 0]
if null_after.empty:
    print("\nAll nulls resolved. Ready for train/test split.")
else:
    print(f"\nRemaining nulls after imputation:\n{null_after}")

In [ ]:
## Save feature dataset before train/test split
file_path = ARTIFACT_DIR / "final_ml_df_cleaned.parquet"
print(file_path)
final_ml_df.to_parquet(file_path, engine="pyarrow", compression="snappy", index=False)
print("feature dataset saved successfully")


In [ ]:
## Temporal Train/Test Split

# Sort explicitly before splitting: older orders train, newer orders test.
final_ml_df = final_ml_df.sort_values("order_purchase_timestamp").reset_index(drop=True)

split_idx = int(len(final_ml_df) * 0.80)
train_df = final_ml_df.iloc[:split_idx].copy()
test_df = final_ml_df.iloc[split_idx:].copy()

# Fill cold-start rolling features with train-only medians to avoid test leakage.
for col in cold_start_cols:
    if col in train_df.columns:
        median_value = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_value)
        test_df[col] = test_df[col].fillna(median_value)

# Fill product dimensions with train-only global medians, then recompute derived physical features.
for col in dim_cols:
    if col in train_df.columns:
        median_value = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_value)
        test_df[col] = test_df[col].fillna(median_value)

for df in [train_df, test_df]:
    if {"product_length_cm", "product_width_cm", "product_height_cm"}.issubset(df.columns):
        df["product_volume"] = (
            df["product_length_cm"] * df["product_width_cm"] * df["product_height_cm"]
        )
    if {"product_weight_g", "product_volume"}.issubset(df.columns):
        df["weight_per_volume"] = (df["product_weight_g"] / df["product_volume"]).replace([np.inf, -np.inf], np.nan)

if "weight_per_volume" in train_df.columns:
    weight_per_volume_median = train_df["weight_per_volume"].median()
    train_df["weight_per_volume"] = train_df["weight_per_volume"].fillna(weight_per_volume_median)
    test_df["weight_per_volume"] = test_df["weight_per_volume"].fillna(weight_per_volume_median)

print(f"\nTrain: {len(train_df):,} rows")
print(f"  Date range: {train_df['order_purchase_timestamp'].min().date()} -> {train_df['order_purchase_timestamp'].max().date()}")
print(f"  Delay rate: {train_df['is_delayed'].mean():.2%}")

print(f"\nTest:  {len(test_df):,} rows")
print(f"  Date range: {test_df['order_purchase_timestamp'].min().date()} -> {test_df['order_purchase_timestamp'].max().date()}")
print(f"  Delay rate: {test_df['is_delayed'].mean():.2%}")


In [ ]:
for col in final_ml_df.columns:
    print(col) 

In [ ]:
## Drop columns not used as model features

leakage_cols = [
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "actual_carrier_delay",
    "actual_carrier_delay_days",
]

id_cols = ["order_id", "seller_id", "product_id", "customer_unique_id"]
target_cols = ["order_delivery_delay", "is_delayed"]
time_cols = ["order_purchase_timestamp"]

drop_cols = id_cols + leakage_cols + target_cols + time_cols
feature_cols = [col for col in final_ml_df.columns if col not in drop_cols]

x_train = train_df[feature_cols].copy()
x_test = test_df[feature_cols].copy()

# CatBoost can consume categorical columns directly.
cat_features = [col for col in ["product_category_name"] if col in feature_cols]
for col in cat_features:
    x_train[col] = x_train[col].astype(str).fillna("unknown")
    x_test[col] = x_test[col].astype(str).fillna("unknown")

# Regression targets
y_reg_train = train_df["order_delivery_delay"]
y_reg_test = test_df["order_delivery_delay"]

# Classification targets
y_clf_train = train_df["is_delayed"]
y_clf_test = test_df["is_delayed"]

print(f"\nFeature count: {len(feature_cols)}")
print(f"Categorical features: {cat_features}")
print(f"Features: {feature_cols}")


In [ ]:
print(f"Train delay rate: {y_clf_train.mean():.2%}")
print(f"Test  delay rate: {y_clf_test.mean():.2%}")

In [ ]:
# y_reg_train.describe()
x_train.columns

In [ ]:
y_reg_train.describe()

In [ ]:
lower_bound = y_reg_train.quantile(0.01)
upper_bound  = y_reg_train.quantile(0.99)
print(" lower limit in Order delay in training data is ",lower_bound)
print(" higher limit in Order delay in training data is ",upper_bound)

import matplotlib.pyplot as plt 
import seaborn as sns 
plt.figure(figsize=(10,6))
sns.boxplot(x=y_reg_train.clip(-30,60),color='lightcoral')
plt.axvline(y_reg_train.mean(),color='red',linestyle='--')
plt.show()

y_reg_train_w = y_reg_train.clip(lower=lower_bound, upper=upper_bound)
y_reg_test_w  = y_reg_test.clip(lower=lower_bound, upper=upper_bound)



In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    y_reg_train,
    bins=100,
    kde=True
)

plt.xlim(-50, 80)
plt.show()

In [ ]:
## count plot - for class imbalance checking

plt.figure(figsize=(10,6))

class_counts = y_clf_train.value_counts()

plt.pie(
    class_counts,
    labels=['On Time', 'Delayed'],
    colors=['#4CAF50', '#F44336'],
    autopct='%1.1f%%',
    startangle=90
)

plt.title("Class Imbalance")

plt.show()


In [ ]:
# Dataset is E-commerce based so naturally most deliveries  are early or on time /delayed deliveries were rare but are
# of Business importance 
# target distribution was right-skewed with extreme delays

## TEMPORAL DATA

## Temporal means data behavior changes with time , time affect behaviour 

# In our dataset:

# orders comes according to date
# delivery patterns change month wise 
# festivals delay can increase at festival time 
# logistics can improve and worsen improve/worsen 

# So this is:

# Temporal Data

# Since this was temporal E-commerce data, I avoided random splitting to prevent future leakage

# past orders → train
# future orders → test

# I analyzed the delivery-delay target distribution using descriptive statistics-"boxplots, and histograms."

# Observed:

# median around--> -12 days
# most deliveries happened before estimated date
# long positive tail existed
# few extreme delays/outliers

# Then explain:

# "Negative delays were business-valid because many sellers used conservative delivery estimates.""

# Since delayed deliveries are business-important rare events, 
# removing rows would reduce valuable signal and worsen imbalance.""
# so "I used percentile-based Winsorization on the target instead of deleting rows."

# "This stabilized extreme values while preserving dataset size and rare delivery patterns."

# In ecommerce logistics, extreme delays are operationally meaningful, not always noise.

# Aggressive outlier deletion could remove exactly the cases the business cares about.""

# the delayed class represented only ~6% of orders, creating a strong class imbalance problem
# I intentionally avoided Synthetically fake data generation using technique like SMOTE in the initial phase because 
# the dataset had temporal structure and synthetic samples could create unrealistic logistics patterns.

# We will Discuss More about this -----------------


# preferred cost-sensitive learning using class-weighted boosting models such as CatBoost/XGBoost

# scale_pos_weight

# “Since the target was skewed and relationships were highly nonlinear, tree boosting models were more suitable than linear regression.”

# Mention:

# XGBoost
# LightGBM
# CatBoost



## Things to be Studied 

#### Skewness 
#### Mean median Mode -meaning and intrepretation 
#### Quantiles meaning 

#### Winsorization==clipping and masking difference 

#### Why Tree Model 
#### Why not transformation when target is Positive skewed?
#### Yes its is obvious delays are both Positive and negative so Log transformation would not work but still Yeo Johnson transformation can work still 

#### Why not delayed delivery order samples artificially introduced using Sampling techniques Like SMOTE 

#### What is temporal data 

#### Why accuracy is misleading here , best parameters in this case would be for classification will be ?


In [ ]:
lt=[]
for col in x_train:
    lt.extend(x_train.select_dtypes(include=['category','category','string']).columns.to_list())
    
print("Catgorical columns are : ")
for col in lt:
    print(col)


In [ ]:
## To Check whether encoding is needed

print(x_train.dtypes.value_counts())
print(x_train.select_dtypes(include='object').columns.tolist())
# Should return empty list []

In [ ]:
# ==========================================
# CATBOOST REGRESSOR
# ==========================================

from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==========================================
# CREATE MODEL
# ==========================================

cat_reg = CatBoostRegressor(

    iterations=1000,

    learning_rate=0.05,

    depth=8,

    loss_function='RMSE',

    eval_metric='RMSE',

    random_seed=42,

    verbose=100
)

# ==========================================
# TRAIN MODEL
# ==========================================

cat_reg.fit(

    x_train,
    y_reg_train,

    eval_set=(x_test, y_reg_test),

    cat_features=cat_features,

    use_best_model=True
)

# ==========================================
# PREDICTIONS
# ==========================================

y_pred = cat_reg.predict(x_test)

# ==========================================
# EVALUATION
# ==========================================

mae = mean_absolute_error(y_reg_test, y_pred)

mse = mean_squared_error(y_reg_test, y_pred)

rmse = mse ** 0.5

r2 = r2_score(y_reg_test, y_pred)

# ==========================================
# RESULTS
# ==========================================

print("=" * 50)

print("REGRESSION RESULTS")

print("=" * 50)

print(f"MAE  : {mae:.4f}")

print(f"RMSE : {rmse:.4f}")

print(f"R2   : {r2:.4f}")

In [ ]:
# ==========================================
# CATBOOST CLASSIFIER
# ==========================================

from catboost import CatBoostClassifier

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (

    classification_report,

    confusion_matrix,

    roc_auc_score,

    average_precision_score
)

import numpy as np

# ==========================================
# COMPUTE CLASS WEIGHTS
# ==========================================

classes = np.unique(y_clf_train)

weights = compute_class_weight(

    class_weight='balanced',

    classes=classes,

    y=y_clf_train
)

class_weights = dict(zip(classes, weights))

print("Class Weights:", class_weights)

# ==========================================
# CREATE MODEL
# ==========================================

cat_clf = CatBoostClassifier(

    iterations=1000,

    learning_rate=0.05,

    depth=6,

    loss_function='Logloss',

    eval_metric='F1',

    class_weights=class_weights,

    random_seed=42,

    verbose=100
)

# ==========================================
# TRAIN MODEL
# ==========================================

cat_clf.fit(

    x_train,
    y_clf_train,

    eval_set=(x_test, y_clf_test),

    cat_features=cat_features,

    use_best_model=True
)

# ==========================================
# PREDICTIONS
# ==========================================

y_pred = cat_clf.predict(x_test)

y_prob = cat_clf.predict_proba(x_test)[:, 1]

# ==========================================
# EVALUATION
# ==========================================

print("=" * 50)

print("CLASSIFICATION REPORT")

print("=" * 50)

print(classification_report(y_clf_test, y_pred))

# ------------------------------------------

print("=" * 50)

print("CONFUSION MATRIX")

print("=" * 50)

print(confusion_matrix(y_clf_test, y_pred))

# ------------------------------------------

roc_auc = roc_auc_score(y_clf_test, y_prob)

pr_auc = average_precision_score(y_clf_test, y_prob)

print("=" * 50)

print(f"ROC-AUC : {roc_auc:.4f}")

print(f"PR-AUC  : {pr_auc:.4f}")